In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("E2E_Pipeline").getOrCreate()
from pyspark.sql.functions import *

In [0]:

data = [
  (1,"C001","Laptop","50000","2024-01-01"),
  (2,"C002","Mobile",None,"2024-01-02"),
  (3,"C003","Tablet","20000","2024-01-03"),
  (4,"C004","Laptop","55000","2024-01-04"),
  (5,"C005","Headphones",None,"2024-01-05"),
  
]
columns=["order_id","customer_id","Product","amount","updated_date"]
df = spark.createDataFrame(data,columns)

In [0]:
df.display()

####Task1 :Handle Null's

In [0]:
# replace null amount with 1000
df=df.fillna({"amount":"1000"})
df.display()

#### Task2 : cast columns

In [0]:
# convert amount->integer and updated date->date
df = df.withColumn("amount", df.amount.cast("int")).withColumn("updated_date", df.updated_date.cast("date"))
display(df)
df.printSchema()

In [0]:
df=df.withColumn("bonus",col("amount")*0.1)
df.display()

In [0]:
df=df.withColumn("amount_category",when(col("amount")>20000,"High").otherwise("low"))
df.display()

#### Task4 apply udf

In [0]:
def amount_bucket(amount):
  if amount<10000:
    return "Low";
  elif ((amount>10000) & (amount<30000)):
    return "Medium";
  else:
    return "High";

In [0]:
my_func=udf(amount_bucket)


In [0]:
from pyspark.sql.functions import *
df = df.withColumn("amount_bucket", my_func(col("amount")))
display(df)

#### Task 5 Full load

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/target")
display(df)

#### Task 6 Incremental load

In [0]:
existing_df=spark.read.parquet("/Volumes/workspace/default/target")
display(existing_df)

In [0]:
last_loaded_date = df.select(max("updated_date")).collect()[0][0]

print("Last Loaded Date:", last_loaded_date)

In [0]:
data2=[
    (3, "C003", "Tablet", 22000, "2024-01-06"),  # updated
    (4, "C004", "Laptop", 56000, "2024-01-06"),  # updated
    (6, "C006", "Camera", 30000, "2024-01-06"),  # new
    (7, "C007", "Mobile", 18000, "2024-01-07"),  # new
    (8, "C008", "Watch", 8000, "2024-01-08")     # new
]
df2=spark.createDataFrame(data2,columns)
df2.display()

In [0]:
df2=df2.withColumn("amount",df2.amount.cast("int")).withColumn("updated_date",df2.updated_date.cast("date"))

In [0]:
df2.display()

In [0]:
new_df = df2
new_df.display()

In [0]:
from pyspark.sql.functions import col

incremental_df = new_df.filter(
    col("updated_date") > last_loaded_date
)
display(incremental_df)

In [0]:
filtered_existing = existing_df.join(
    incremental_df,
    on="order_id",
    how="left_anti"
)
display(filtered_existing)
# Remove older versions of order_id 3 and 4

In [0]:
# Apply same transformations to incremental_df
incremental_df = incremental_df.withColumn("bonus", col("amount") * 0.1)
incremental_df = incremental_df.withColumn("amount_category", when(col("amount") > 20000, "High").otherwise("low"))
incremental_df = incremental_df.withColumn("amount_bucket", my_func(col("amount")))

final_df = filtered_existing.union(incremental_df)
final_df.display()

In [0]:
final_df.write.mode("overwrite").parquet("/Volumes/workspace/default/target")

#### Task7 parameterization